# 🚀 完整推理流程和端到端系统

## 📋 学习目标

本Notebook将带你深入了解LLM推理系统的完整流程，从模型加载到最终输出的端到端实现：

### 🎯 核心内容
1. **模型加载与初始化** - 完整的模型加载流程
2. **LLM引擎架构** - 核心推理引擎的设计与实现
3. **PagedAttention机制** - 高效的注意力计算优化
4. **端到端推理流程** - 从输入到输出的完整处理链
5. **性能监控与优化** - 实时性能跟踪和优化策略
6. **分布式推理** - 多GPU和多节点推理架构

### 🔧 技术要点
- 模型权重管理和内存优化
- 动态批处理和请求调度
- KV Cache的高效管理
- 流式输出和实时响应
- 错误处理和系统恢复

让我们开始这个激动人心的学习之旅！🎉

## 🔧 环境设置

首先设置我们的实验环境，包括必要的依赖和工具。

In [ ]:
# 检测是否在Colab环境中运行
try:
    import google.colab
    IN_COLAB = True
    print("🔍 检测到Google Colab环境")
except ImportError:
    IN_COLAB = False
    print("🔍 检测到本地环境")

# 如果在Colab中，克隆项目仓库
if IN_COLAB:
    print("📥 正在克隆nano-vLLM项目...")
    !git clone https://github.com/your-username/nano-vllm-learning.git
    %cd nano-vllm-learning
    print("✅ 项目克隆完成")

# 安装必要的依赖
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# 基础依赖
required_packages = [
    "torch",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",
    "psutil",
    "transformers"
]

print("📦 安装必要的依赖包...")
for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} 已安装")
    except ImportError:
        print(f"📥 正在安装 {package}...")
        install_package(package)
        print(f"✅ {package} 安装完成")

print("\n🎉 环境设置完成！")

In [ ]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple, Any, Union
from dataclasses import dataclass, field
from enum import Enum
import time
import threading
import queue
import json
import logging
from collections import defaultdict, deque
import psutil
import gc
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 设置绘图样式
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

print("📚 库导入完成")
print(f"🔥 PyTorch版本: {torch.__version__}")
print(f"🎯 CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 🏗️ 模型加载与初始化

让我们从模型加载开始，这是整个推理系统的基础。我们将实现一个完整的模型加载器，包括权重管理、内存优化和配置处理。

In [ ]:
@dataclass
class ModelConfig:
    """模型配置类"""
    model_name: str = "nano-llm-7b"
    vocab_size: int = 32000
    hidden_size: int = 4096
    num_layers: int = 32
    num_attention_heads: int = 32
    num_key_value_heads: int = 32  # GQA支持
    intermediate_size: int = 11008
    max_position_embeddings: int = 4096
    rms_norm_eps: float = 1e-6
    rope_theta: float = 10000.0
    attention_dropout: float = 0.0
    
    # 推理配置
    max_batch_size: int = 32
    max_seq_len: int = 2048
    dtype: torch.dtype = torch.float16
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 内存配置
    kv_cache_dtype: torch.dtype = torch.float16
    block_size: int = 16  # PagedAttention块大小
    max_num_blocks: int = 1024
    
    def __post_init__(self):
        """配置后处理"""
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.num_kv_heads = self.num_key_value_heads or self.num_attention_heads
        self.kv_head_dim = self.hidden_size // self.num_kv_heads
        
        # 计算内存需求
        self.estimate_memory_usage()
    
    def estimate_memory_usage(self):
        """估算内存使用量"""
        # 模型参数内存 (GB)
        param_memory = (
            # Embedding
            self.vocab_size * self.hidden_size +
            # Transformer layers
            self.num_layers * (
                # Attention weights
                self.hidden_size * (self.hidden_size + 2 * self.num_kv_heads * self.kv_head_dim) +
                # MLP weights
                self.hidden_size * self.intermediate_size * 2 +
                # Layer norms
                self.hidden_size * 2
            ) +
            # Final layer norm
            self.hidden_size
        ) * 2 / (1024**3)  # 2 bytes per parameter (fp16)
        
        # KV Cache内存 (GB)
        kv_cache_memory = (
            self.max_batch_size * self.max_seq_len * 
            self.num_layers * 2 * self.num_kv_heads * self.kv_head_dim * 2
        ) / (1024**3)
        
        self.param_memory_gb = param_memory
        self.kv_cache_memory_gb = kv_cache_memory
        self.total_memory_gb = param_memory + kv_cache_memory
        
        print(f"📊 内存估算:")
        print(f"   模型参数: {param_memory:.2f} GB")
        print(f"   KV Cache: {kv_cache_memory:.2f} GB")
        print(f"   总计: {self.total_memory_gb:.2f} GB")

# 创建模型配置
config = ModelConfig()
print("✅ 模型配置创建完成")

In [ ]:
class ModelLoader:
    """模型加载器 - 负责模型的加载、初始化和内存管理"""
    
    def __init__(self, config: ModelConfig):
        self.config = config
        self.device = torch.device(config.device)
        self.logger = self._setup_logger()
        
    def _setup_logger(self):
        """设置日志记录器"""
        logger = logging.getLogger("ModelLoader")
        logger.setLevel(logging.INFO)
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            handler.setFormatter(formatter)
            logger.addHandler(handler)
        return logger
    
    def load_model_weights(self, model_path: Optional[str] = None) -> Dict[str, torch.Tensor]:
        """加载模型权重"""
        self.logger.info("🔄 开始加载模型权重...")
        
        if model_path:
            # 从文件加载真实权重
            try:
                weights = torch.load(model_path, map_location='cpu')
                self.logger.info(f"✅ 从 {model_path} 加载权重成功")
                return weights
            except Exception as e:
                self.logger.warning(f"⚠️ 加载权重失败: {e}，使用随机权重")
        
        # 生成随机权重用于演示
        weights = self._generate_random_weights()
        self.logger.info("✅ 生成随机权重完成")
        return weights
    
    def _generate_random_weights(self) -> Dict[str, torch.Tensor]:
        """生成随机权重用于演示"""
        weights = {}
        config = self.config
        
        # Token embedding
        weights['embed_tokens.weight'] = torch.randn(
            config.vocab_size, config.hidden_size, dtype=config.dtype
        )
        
        # Transformer layers
        for layer_idx in range(config.num_layers):
            prefix = f'layers.{layer_idx}'
            
            # Self-attention weights
            weights[f'{prefix}.self_attn.q_proj.weight'] = torch.randn(
                config.hidden_size, config.hidden_size, dtype=config.dtype
            )
            weights[f'{prefix}.self_attn.k_proj.weight'] = torch.randn(
                config.num_kv_heads * config.kv_head_dim, config.hidden_size, dtype=config.dtype
            )
            weights[f'{prefix}.self_attn.v_proj.weight'] = torch.randn(
                config.num_kv_heads * config.kv_head_dim, config.hidden_size, dtype=config.dtype
            )
            weights[f'{prefix}.self_attn.o_proj.weight'] = torch.randn(
                config.hidden_size, config.hidden_size, dtype=config.dtype
            )
            
            # MLP weights
            weights[f'{prefix}.mlp.gate_proj.weight'] = torch.randn(
                config.intermediate_size, config.hidden_size, dtype=config.dtype
            )
            weights[f'{prefix}.mlp.up_proj.weight'] = torch.randn(
                config.intermediate_size, config.hidden_size, dtype=config.dtype
            )
            weights[f'{prefix}.mlp.down_proj.weight'] = torch.randn(
                config.hidden_size, config.intermediate_size, dtype=config.dtype
            )
            
            # Layer norms
            weights[f'{prefix}.input_layernorm.weight'] = torch.ones(
                config.hidden_size, dtype=config.dtype
            )
            weights[f'{prefix}.post_attention_layernorm.weight'] = torch.ones(
                config.hidden_size, dtype=config.dtype
            )
        
        # Final layer norm and output projection
        weights['norm.weight'] = torch.ones(config.hidden_size, dtype=config.dtype)
        weights['lm_head.weight'] = torch.randn(
            config.vocab_size, config.hidden_size, dtype=config.dtype
        )
        
        return weights
    
    def optimize_memory_layout(self, weights: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        """优化内存布局"""
        self.logger.info("🔧 优化内存布局...")
        
        optimized_weights = {}
        
        for name, weight in weights.items():
            # 确保权重是连续的
            if not weight.is_contiguous():
                weight = weight.contiguous()
            
            # 移动到目标设备
            weight = weight.to(self.device)
            
            # 对于大权重矩阵，考虑使用更高效的存储格式
            if 'proj.weight' in name and weight.numel() > 1000000:
                # 可以在这里实现权重量化或其他优化
                pass
            
            optimized_weights[name] = weight
        
        self.logger.info("✅ 内存布局优化完成")
        return optimized_weights
    
    def validate_weights(self, weights: Dict[str, torch.Tensor]) -> bool:
        """验证权重的完整性"""
        self.logger.info("🔍 验证权重完整性...")
        
        required_keys = [
            'embed_tokens.weight',
            'norm.weight',
            'lm_head.weight'
        ]
        
        # 检查必需的权重
        for key in required_keys:
            if key not in weights:
                self.logger.error(f"❌ 缺少必需的权重: {key}")
                return False
        
        # 检查层权重
        for layer_idx in range(self.config.num_layers):
            layer_keys = [
                f'layers.{layer_idx}.self_attn.q_proj.weight',
                f'layers.{layer_idx}.self_attn.k_proj.weight',
                f'layers.{layer_idx}.self_attn.v_proj.weight',
                f'layers.{layer_idx}.self_attn.o_proj.weight',
                f'layers.{layer_idx}.mlp.gate_proj.weight',
                f'layers.{layer_idx}.mlp.up_proj.weight',
                f'layers.{layer_idx}.mlp.down_proj.weight',
                f'layers.{layer_idx}.input_layernorm.weight',
                f'layers.{layer_idx}.post_attention_layernorm.weight'
            ]
            
            for key in layer_keys:
                if key not in weights:
                    self.logger.error(f"❌ 缺少层权重: {key}")
                    return False
        
        # 检查权重形状
        embed_weight = weights['embed_tokens.weight']
        if embed_weight.shape != (self.config.vocab_size, self.config.hidden_size):
            self.logger.error(f"❌ 嵌入权重形状错误: {embed_weight.shape}")
            return False
        
        self.logger.info("✅ 权重验证通过")
        return True
    
    def get_memory_usage(self) -> Dict[str, float]:
        """获取内存使用情况"""
        if torch.cuda.is_available():
            gpu_memory = torch.cuda.memory_allocated() / 1024**3
            gpu_cached = torch.cuda.memory_reserved() / 1024**3
        else:
            gpu_memory = gpu_cached = 0.0
        
        cpu_memory = psutil.Process().memory_info().rss / 1024**3
        
        return {
            'gpu_allocated': gpu_memory,
            'gpu_cached': gpu_cached,
            'cpu_memory': cpu_memory
        }

# 创建模型加载器并加载权重
loader = ModelLoader(config)
weights = loader.load_model_weights()
weights = loader.optimize_memory_layout(weights)
is_valid = loader.validate_weights(weights)

# 显示内存使用情况
memory_usage = loader.get_memory_usage()
print(f"\n💾 内存使用情况:")
print(f"   GPU已分配: {memory_usage['gpu_allocated']:.2f} GB")
print(f"   GPU缓存: {memory_usage['gpu_cached']:.2f} GB")
print(f"   CPU内存: {memory_usage['cpu_memory']:.2f} GB")

print(f"\n✅ 模型加载完成，权重验证: {'通过' if is_valid else '失败'}")

## 🧠 PagedAttention机制实现

PagedAttention是现代LLM推理系统的核心技术，它通过分块管理KV Cache来大幅提高内存效率。让我们实现一个完整的PagedAttention系统。

In [ ]:
class PagedAttentionManager:
    """PagedAttention管理器 - 实现高效的KV Cache管理"""
    
    def __init__(self, config: ModelConfig):
        self.config = config
        self.block_size = config.block_size
        self.max_num_blocks = config.max_num_blocks
        self.num_layers = config.num_layers
        self.num_kv_heads = config.num_kv_heads
        self.head_dim = config.kv_head_dim
        self.device = torch.device(config.device)
        
        # 初始化物理块存储
        self._init_physical_blocks()
        
        # 块管理
        self.free_blocks = set(range(self.max_num_blocks))
        self.allocated_blocks = {}
        
        # 序列到块的映射
        self.sequence_blocks = defaultdict(list)
        
        print(f"🧠 PagedAttention管理器初始化完成")
        print(f"   块大小: {self.block_size}")
        print(f"   最大块数: {self.max_num_blocks}")
        print(f"   总KV Cache容量: {self._calculate_total_capacity()} tokens")
    
    def _init_physical_blocks(self):
        """初始化物理块存储"""
        # Key blocks: [num_blocks, num_layers, num_kv_heads, block_size, head_dim]
        self.key_blocks = torch.zeros(
            self.max_num_blocks,
            self.num_layers,
            self.num_kv_heads,
            self.block_size,
            self.head_dim,
            dtype=self.config.kv_cache_dtype,
            device=self.device
        )
        
        # Value blocks: [num_blocks, num_layers, num_kv_heads, block_size, head_dim]
        self.value_blocks = torch.zeros(
            self.max_num_blocks,
            self.num_layers,
            self.num_kv_heads,
            self.block_size,
            self.head_dim,
            dtype=self.config.kv_cache_dtype,
            device=self.device
        )
    
    def _calculate_total_capacity(self) -> int:
        """计算总KV Cache容量"""
        return self.max_num_blocks * self.block_size
    
    def allocate_sequence(self, sequence_id: str, sequence_length: int) -> List[int]:
        """为序列分配块"""
        num_blocks_needed = (sequence_length + self.block_size - 1) // self.block_size
        
        if len(self.free_blocks) < num_blocks_needed:
            raise RuntimeError(f"内存不足：需要 {num_blocks_needed} 块，但只有 {len(self.free_blocks)} 块可用")
        
        # 分配块
        allocated_block_ids = []
        for _ in range(num_blocks_needed):
            block_id = self.free_blocks.pop()
            allocated_block_ids.append(block_id)
            self.allocated_blocks[block_id] = sequence_id
        
        self.sequence_blocks[sequence_id] = allocated_block_ids
        
        print(f"📦 为序列 {sequence_id} 分配了 {num_blocks_needed} 个块: {allocated_block_ids}")
        return allocated_block_ids
    
    def free_sequence(self, sequence_id: str):
        """释放序列的所有块"""
        if sequence_id not in self.sequence_blocks:
            return
        
        block_ids = self.sequence_blocks[sequence_id]
        
        # 释放块
        for block_id in block_ids:
            self.free_blocks.add(block_id)
            del self.allocated_blocks[block_id]
        
        del self.sequence_blocks[sequence_id]
        
        print(f"🗑️ 释放序列 {sequence_id} 的 {len(block_ids)} 个块")
    
    def store_kv_cache(self, sequence_id: str, layer_idx: int, 
                      key_states: torch.Tensor, value_states: torch.Tensor,
                      start_pos: int = 0):
        """存储KV Cache到分页内存"""
        if sequence_id not in self.sequence_blocks:
            raise ValueError(f"序列 {sequence_id} 未分配块")
        
        block_ids = self.sequence_blocks[sequence_id]
        seq_len = key_states.shape[1]  # [batch_size, seq_len, num_heads, head_dim]
        
        # 将KV states存储到对应的块中
        for i, token_idx in enumerate(range(start_pos, start_pos + seq_len)):
            block_idx = token_idx // self.block_size
            offset_in_block = token_idx % self.block_size
            
            if block_idx < len(block_ids):
                physical_block_id = block_ids[block_idx]
                
                # 存储key和value
                self.key_blocks[physical_block_id, layer_idx, :, offset_in_block, :] = \
                    key_states[0, i, :, :].to(self.config.kv_cache_dtype)
                self.value_blocks[physical_block_id, layer_idx, :, offset_in_block, :] = \
                    value_states[0, i, :, :].to(self.config.kv_cache_dtype)
    
    def retrieve_kv_cache(self, sequence_id: str, layer_idx: int, 
                         max_length: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """从分页内存检索KV Cache"""
        if sequence_id not in self.sequence_blocks:
            raise ValueError(f"序列 {sequence_id} 未分配块")
        
        block_ids = self.sequence_blocks[sequence_id]
        
        # 计算需要检索的token数量
        total_tokens = min(max_length, len(block_ids) * self.block_size)
        
        # 初始化输出张量
        key_cache = torch.zeros(
            1, total_tokens, self.num_kv_heads, self.head_dim,
            dtype=self.config.kv_cache_dtype, device=self.device
        )
        value_cache = torch.zeros(
            1, total_tokens, self.num_kv_heads, self.head_dim,
            dtype=self.config.kv_cache_dtype, device=self.device
        )
        
        # 从块中检索数据
        for token_idx in range(total_tokens):
            block_idx = token_idx // self.block_size
            offset_in_block = token_idx % self.block_size
            
            if block_idx < len(block_ids):
                physical_block_id = block_ids[block_idx]
                
                key_cache[0, token_idx, :, :] = \
                    self.key_blocks[physical_block_id, layer_idx, :, offset_in_block, :]
                value_cache[0, token_idx, :, :] = \
                    self.value_blocks[physical_block_id, layer_idx, :, offset_in_block, :]
        
        return key_cache, value_cache
    
    def get_memory_stats(self) -> Dict[str, Any]:
        """获取内存统计信息"""
        total_blocks = self.max_num_blocks
        used_blocks = len(self.allocated_blocks)
        free_blocks = len(self.free_blocks)
        utilization = used_blocks / total_blocks * 100
        
        # 计算内存使用量
        block_memory_mb = (
            self.num_layers * self.num_kv_heads * self.block_size * 
            self.head_dim * 2 * 2  # 2 for key+value, 2 bytes for fp16
        ) / (1024 * 1024)
        
        total_memory_mb = total_blocks * block_memory_mb
        used_memory_mb = used_blocks * block_memory_mb
        
        return {
            'total_blocks': total_blocks,
            'used_blocks': used_blocks,
            'free_blocks': free_blocks,
            'utilization_percent': utilization,
            'total_memory_mb': total_memory_mb,
            'used_memory_mb': used_memory_mb,
            'block_memory_mb': block_memory_mb,
            'sequences': len(self.sequence_blocks)
        }
    
    def visualize_memory_layout(self):
        """可视化内存布局"""
        stats = self.get_memory_stats()
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # 块使用情况饼图
        labels = ['已使用', '空闲']
        sizes = [stats['used_blocks'], stats['free_blocks']]
        colors = ['#ff9999', '#66b3ff']
        
        ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        ax1.set_title(f'内存块使用情况\n总计: {stats["total_blocks"]} 块')
        
        # 内存使用条形图
        categories = ['总内存', '已使用', '空闲']
        memory_values = [
            stats['total_memory_mb'],
            stats['used_memory_mb'],
            stats['total_memory_mb'] - stats['used_memory_mb']
        ]
        
        bars = ax2.bar(categories, memory_values, color=['#ffcc99', '#ff9999', '#66b3ff'])
        ax2.set_ylabel('内存 (MB)')
        ax2.set_title('内存使用统计')
        
        # 添加数值标签
        for bar, value in zip(bars, memory_values):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01*max(memory_values),
                    f'{value:.1f}MB', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        # 打印详细统计
        print(f"\n📊 PagedAttention内存统计:")
        print(f"   总块数: {stats['total_blocks']}")
        print(f"   已使用: {stats['used_blocks']} ({stats['utilization_percent']:.1f}%)")
        print(f"   空闲: {stats['free_blocks']}")
        print(f"   活跃序列: {stats['sequences']}")
        print(f"   总内存: {stats['total_memory_mb']:.1f} MB")
        print(f"   已用内存: {stats['used_memory_mb']:.1f} MB")
        print(f"   单块内存: {stats['block_memory_mb']:.2f} MB")

# 创建PagedAttention管理器
paged_attention = PagedAttentionManager(config)

# 演示分配和使用
print("\n🧪 演示PagedAttention使用:")

# 分配几个序列
seq_lengths = [128, 256, 512, 64]
sequence_ids = []

for i, length in enumerate(seq_lengths):
    seq_id = f"seq_{i}"
    sequence_ids.append(seq_id)
    try:
        paged_attention.allocate_sequence(seq_id, length)
    except RuntimeError as e:
        print(f"❌ 分配失败: {e}")
        break

# 可视化内存布局
paged_attention.visualize_memory_layout()